# Swarm trajectory recorder — visualization notebook

Loads one `*_traj.json` produced by `SwarmTrajectoryRecorder.cs` and renders a set of plots:

1. **Top-down trajectory** with obstacles overlaid + crash markers
2. **Swarm health** over time (main group, disconnected, subnetwork count, cumulative crashes)
3. **Crash timeline** (when, which drone, embodied or not)
4. **Inputs** (fused movement / spread / rotation + per-source breakdown)
5. **Gap geometry** — subnetwork centroids while the swarm is split
6. **Run summary** — text card with the headline numbers

Run cells top-to-bottom, or jump straight to the plot you care about — every plot cell is independent once the *Load* cell has run.

**Requirements:** `pip install matplotlib numpy`

## Setup

In [ ]:
from __future__ import annotations
import json
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

## Load a run
Point `JSON_PATH` at the file you want. Default scans `../SoundMappingUnity/Assets/Trajectories/` and grabs the most recent one.

In [ ]:
# --- Pick a file ---
# Either paste an absolute path here:
JSON_PATH: Path | None = None

# ...or auto-pick the most recent traj.json under the Editor save folder.
if JSON_PATH is None:
    search_root = Path("../SoundMappingUnity/Assets/Trajectories")
    candidates = sorted(search_root.rglob("*_traj.json"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError(f"No *_traj.json files under {search_root.resolve()}")
    JSON_PATH = candidates[0]

JSON_PATH = Path(JSON_PATH)
print(f"Loading: {JSON_PATH}")

with open(JSON_PATH, "r", encoding="utf-8") as f:
    log = json.load(f)

# Reference time = start of the first sample, so every time axis is run-relative.
def _t0(log: dict) -> float:
    sf = log.get("swarmFrames") or []
    if sf:
        return sf[0]["t"]
    for d in log.get("trajectories", []):
        if d.get("frames"):
            return d["frames"][0]["t"]
    return 0.0

T0 = _t0(log)
print(
    f"PID={log.get('pid','?')}  scene={log.get('scene','?')}  hap={log.get('haptics','?')}  order={log.get('order','?')}\n"
    f"drones={len(log.get('trajectories', []))}  swarmFrames={len(log.get('swarmFrames', []))}  "
    f"subnetRows={len(log.get('subnetworks', []))}  crashes={log.get('crashCount', 0)}  "
    f"collectibles={log.get('collectiblesPickedUp', 0)}/{log.get('totalCollectibles', 0)}  "
    f"elapsed={log.get('elapsedTime', 0):.2f}s"
)

## Helpers (obstacle shapes for top-down view)

In [ ]:
def _yaw_from_quat(qw, qx, qy, qz):
    """Yaw (rotation about Unity's Y axis) — what we need for top-down XZ."""
    return float(np.arctan2(2.0 * (qw * qy + qx * qz), 1.0 - 2.0 * (qy * qy + qx * qx)))

def obstacle_patches_xz(obstacles, **patch_kwargs):
    """Build matplotlib patches for a top-down XZ view of the obstacle list.
    Spheres + cylinders project to circles; boxes to rotated rectangles."""
    defaults = dict(facecolor="#888888", alpha=0.35, edgecolor="black", linewidth=0.8)
    defaults.update(patch_kwargs)
    out = []
    for o in obstacles or []:
        t = o.get("type", "")
        cx, cz = float(o["cx"]), float(o["cz"])
        if t in ("Sphere", "Cylinder"):
            out.append(mpatches.Circle((cx, cz), float(o.get("radius", 0.5)), **defaults))
        elif t == "Box":
            sx, sz = float(o.get("sx", 1.0)), float(o.get("sz", 1.0))
            yaw = _yaw_from_quat(o.get("qw", 1.0), o.get("qx", 0.0), o.get("qy", 0.0), o.get("qz", 0.0))
            half = np.array([[-sx/2, -sz/2], [sx/2, -sz/2], [sx/2, sz/2], [-sx/2, sz/2]])
            c, s = np.cos(yaw), np.sin(yaw)
            R = np.array([[c, -s], [s, c]])
            corners = (R @ half.T).T + np.array([cx, cz])
            out.append(mpatches.Polygon(corners, **defaults))
        else:
            out.append(mpatches.Circle((cx, cz), float(o.get("radius", 0.5)), linestyle=":", **defaults))
    return out

## 1. Top-down trajectory + obstacles + crashes
Embodied drone is highlighted in red with start (○) and end (□) markers; crashes are black ✕.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

for p in obstacle_patches_xz(log.get("obstacles", [])):
    ax.add_patch(p)

trajs = log.get("trajectories", [])
embodied_id = log.get("embodiedId", None)
cmap = plt.get_cmap("tab20")

for i, drone in enumerate(trajs):
    frames = drone.get("frames", [])
    if not frames:
        continue
    xs = np.fromiter((f["x"] for f in frames), dtype=float, count=len(frames))
    zs = np.fromiter((f["z"] for f in frames), dtype=float, count=len(frames))
    if drone.get("id") == embodied_id:
        ax.plot(xs, zs, color="red", lw=2.0, alpha=0.95, zorder=5, label=f"embodied ({drone.get('name','?')})")
        ax.scatter([xs[0]], [zs[0]], marker="o", s=80, facecolor="white", edgecolor="red", zorder=7, label="embodied start")
        ax.scatter([xs[-1]], [zs[-1]], marker="s", s=80, facecolor="white", edgecolor="red", zorder=7, label="embodied end")
    else:
        ax.plot(xs, zs, color=cmap(i % 20), lw=0.7, alpha=0.7)

crashes = log.get("crashes", [])
if crashes:
    ax.scatter([c["x"] for c in crashes], [c["z"] for c in crashes],
               marker="x", s=90, color="black", linewidths=2.2, zorder=6,
               label=f"crashes ({len(crashes)})")

ax.set_aspect("equal", adjustable="datalim")
ax.set_xlabel("X (m)"); ax.set_ylabel("Z (m)")
ax.set_title(f"Trajectory + obstacles + crashes — PID {log.get('pid','?')} ({log.get('haptics','?')}/{log.get('order','?')})")
ax.legend(loc="best", fontsize=8, framealpha=0.85); ax.grid(True, alpha=0.2)
plt.show()

## 2. Swarm health over time
Main group size, disconnected drones, subnetwork count (gap count = `nSub - 1`), cumulative crashes. Trial start/end are dotted vertical lines; the trial window is faintly shaded.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sf = log.get("swarmFrames", [])
if not sf:
    ax.text(0.5, 0.5, "no swarmFrames recorded", ha="center", va="center", transform=ax.transAxes)
else:
    t  = np.array([f["t"] - T0 for f in sf])
    ax.plot(t, [f["nMain"]  for f in sf], label="main group size",         color="tab:green",  lw=1.6)
    ax.plot(t, [f["nDisc"]  for f in sf], label="disconnected drones",      color="tab:orange", lw=1.4)
    ax.plot(t, [f["nSub"]   for f in sf], label="subnetworks (gaps = -1)",  color="tab:blue",   lw=1.2)
    ax.plot(t, [f["nCrash"] for f in sf], label="cumulative crashes",       color="tab:red",    lw=1.2, linestyle="--")
    for trial in log.get("trials", []):
        ts = trial["startGameTime"] - T0
        ax.axvline(ts, color="black", linestyle=":", alpha=0.4)
        if trial.get("endGameTime", 0) > 0:
            te = trial["endGameTime"] - T0
            ax.axvline(te, color="black", linestyle=":", alpha=0.4)
            ax.axvspan(ts, te, alpha=0.04, color="black")
    ax.set_xlabel("time since start (s)"); ax.set_ylabel("count")
    ax.set_title("Swarm health over time")
    ax.legend(loc="best", framealpha=0.9); ax.grid(True, alpha=0.3)
plt.show()

## 3. Crash timeline
x = time since start, y = drone id. Red = the embodied drone at crash time.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
if not crashes:
    ax.text(0.5, 0.5, "no crashes recorded", ha="center", va="center", transform=ax.transAxes)
else:
    ts = [c["t"] - T0 for c in crashes]
    ids = [c["droneId"] for c in crashes]
    colors = ["red" if c.get("embodied", 0) else "black" for c in crashes]
    ax.scatter(ts, ids, c=colors, marker="x", s=90, linewidths=2.2)
    ax.set_xlabel("time since start (s)"); ax.set_ylabel("drone id")
    ax.set_title(f"Crash timeline ({len(crashes)} total; red = embodied at crash)")
    ax.grid(True, alpha=0.3)
plt.show()

## 4. Inputs over time
Fused signals from `InputFusionManager` plus per-source rotation rates when available.

In [ ]:
inputs = log.get("inputs", [])
if not inputs:
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.text(0.5, 0.5, "no inputs recorded (logInputs disabled?)", ha="center", va="center", transform=ax.transAxes)
    plt.show()
else:
    fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
    t = np.array([f["t"] - T0 for f in inputs])

    axes[0].plot(t, [f["fmx"] for f in inputs], label="x",          color="tab:blue")
    axes[0].plot(t, [f["fmy"] for f in inputs], label="y (height)", color="tab:green")
    axes[0].plot(t, [f["fmz"] for f in inputs], label="z",          color="tab:red")
    axes[0].set_ylabel("SwarmMovement"); axes[0].legend(loc="best"); axes[0].grid(True, alpha=0.3)

    axes[1].plot(t, [f["fs"] for f in inputs], color="tab:purple", label="fused")
    axes[1].set_ylabel("SwarmSpread"); axes[1].grid(True, alpha=0.3)

    axes[2].plot(t, [f["fr"] for f in inputs], color="tab:orange", label="fused")
    axes[2].set_ylabel("CameraRotation"); axes[2].grid(True, alpha=0.3)

    imu_y  = np.array([f.get("imu_y",  np.nan) for f in inputs], dtype=float)
    trad_r = np.array([f.get("trad_r", np.nan) for f in inputs], dtype=float)
    mq_y   = np.array([f.get("mq_y",   np.nan) for f in inputs], dtype=float)
    pose_y = np.array([f.get("pose_y", np.nan) for f in inputs], dtype=float)
    plotted = False
    if np.any(~np.isnan(imu_y)):  axes[3].plot(t, imu_y,  label="IMU yaw rate",     color="tab:cyan");                 plotted = True
    if np.any(~np.isnan(trad_r)): axes[3].plot(t, trad_r, label="traditional R",    color="tab:gray",   lw=0.8);       plotted = True
    if np.any(~np.isnan(mq_y)):   axes[3].plot(t, mq_y,   label="MQ headset yaw",   color="tab:olive",  lw=0.8);       plotted = True
    if np.any(~np.isnan(pose_y)): axes[3].plot(t, pose_y, label="webcam pose yaw",  color="tab:pink",   lw=0.8);       plotted = True
    if plotted: axes[3].legend(loc="best", fontsize=8)
    axes[3].set_ylabel("rotation sources"); axes[3].set_xlabel("time since start (s)"); axes[3].grid(True, alpha=0.3)
    plt.show()

## 5. Gap geometry
Subnetwork centroids overlaid on obstacles, **only when the swarm is actually split**. Marker size ∝ drones in that subnetwork; color encodes time. Distance between same-color markers = how far apart the pieces drifted at that instant.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

for p in obstacle_patches_xz(log.get("obstacles", []), facecolor="#888888", alpha=0.12):
    ax.add_patch(p)

snaps = log.get("subnetworks", [])
by_t: dict[float, list] = {}
for s in snaps:
    by_t.setdefault(s["t"], []).append(s)
split_t = sorted(t for t, lst in by_t.items() if len(lst) > 1)

if not snaps:
    ax.text(0.5, 0.5, "no subnetwork data", ha="center", va="center", transform=ax.transAxes)
elif not split_t:
    ax.text(0.5, 0.5, "swarm never split during this run", ha="center", va="center", transform=ax.transAxes)
else:
    t_lo = split_t[0] - T0
    t_hi = split_t[-1] - T0
    span = max(1e-6, t_hi - t_lo)
    cmap_g = plt.get_cmap("plasma")
    for t in split_t:
        col = cmap_g((t - T0 - t_lo) / span)
        for s in by_t[t]:
            ax.scatter(s["cx"], s["cz"], s=max(15.0, float(s["size"]) * 25.0),
                       facecolor=col, alpha=0.55, edgecolor="black", linewidth=0.4)
    sm = plt.cm.ScalarMappable(cmap=cmap_g, norm=plt.Normalize(vmin=t_lo, vmax=t_hi))
    sm.set_array([])
    plt.colorbar(sm, ax=ax, label="time since start (s)")
    ax.set_title(f"Subnetwork centroids during gaps (n={len(split_t)} split samples; size ∝ drones)")

ax.set_aspect("equal", adjustable="datalim")
ax.set_xlabel("X (m)"); ax.set_ylabel("Z (m)")
plt.show()

## 6. Run summary

In [ ]:
elapsed    = log.get("elapsedTime", 0.0)
picked     = log.get("collectiblesPickedUp", 0)
total      = log.get("totalCollectibles", 0)
n_crashes  = log.get("crashCount", 0)
mean_disc  = float(np.mean([f["nDisc"] for f in sf])) if sf else 0.0
max_sub    = int(np.max([f["nSub"] for f in sf])) if sf else 0

fig, ax = plt.subplots(figsize=(7, 5))
ax.axis("off")
lines = [
    f"PID:        {log.get('pid','?')}    Haptics: {log.get('haptics','?')}    Order: {log.get('order','?')}",
    f"Scene:      {log.get('scene','?')}",
    f"Embodied:   {log.get('embodiedName','?')} (id={log.get('embodiedId','?')})",
    "",
    f"Elapsed time:        {elapsed:.2f} s",
    f"Collectibles:        {picked} / {total}" + (f"  ({100*picked/total:.0f}%)" if total > 0 else ""),
    f"Crashes:             {n_crashes}",
    f"Mean disconnected:   {mean_disc:.2f} drones/frame",
    f"Max subnetworks:     {max_sub}  ({max(0, max_sub - 1)} gaps at peak)",
]
ax.text(0.02, 0.98, "\n".join(lines), family="monospace", fontsize=11, va="top", ha="left", transform=ax.transAxes)
ax.set_title("Run summary")
plt.show()

## (optional) Save all plots to PNG
Run this cell to dump every figure above to a `<jsonname>_plots/` folder next to the JSON. Re-render any cell above first to get its current figure into the cache.

In [ ]:
outdir = JSON_PATH.parent / f"{JSON_PATH.stem}_plots"
outdir.mkdir(parents=True, exist_ok=True)
for i, num in enumerate(plt.get_fignums(), start=1):
    fig = plt.figure(num)
    fig.savefig(outdir / f"{JSON_PATH.stem}_fig{i:02d}.png", dpi=130, bbox_inches="tight")
print(f"Wrote {len(plt.get_fignums())} figures to {outdir}/")